# ⚡ Day 03a: WSGI vs ASGI — How Python Web Servers Actually Work

---

## 🎯 What You'll Master Today
- What WSGI and ASGI are and why they exist
- Synchronous vs asynchronous request handling
- Gunicorn, Uvicorn, and how they serve your app
- Write a minimal WSGI and ASGI app from scratch

---

## 🎭 The Analogy

Imagine a call center:

**WSGI (Synchronous)** = One operator per phone line. Each operator handles ONE call from start to finish. If they're on hold waiting for a database, they just... sit there. Doing nothing.

**ASGI (Asynchronous)** = Smart operators who can juggle multiple calls. While waiting for a database response, they pick up another call. When the DB responds, they switch back. Same number of operators, way more calls handled.

---

## 🏗️ The Big Picture

```
╔══════════════════════════════════════════════════════════════════╗
║                    HOW PYTHON WEB SERVING WORKS                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║   Client → HTTP → Web Server → Gateway Interface → Your App    ║
║                                                                  ║
║   Example:                                                       ║
║   Browser → Nginx → Gunicorn (WSGI) → Flask/Django app         ║
║   Browser → Nginx → Uvicorn (ASGI)  → FastAPI app              ║
║                                                                  ║
║   ┌─────────────────────────────────────────────────────────┐   ║
║   │  WSGI = Web Server Gateway Interface (PEP 3333, 2003) │   ║
║   │  ASGI = Asynchronous Server Gateway Interface (2016)   │   ║
║   └─────────────────────────────────────────────────────────┘   ║
║                                                                  ║
║   They're just STANDARDS — a contract between the server       ║
║   and your application about how to communicate.               ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

## 📦 WSGI — The Synchronous Standard

WSGI is dead simple. Your app is a **callable** that takes two arguments:
- `environ` — a dict with all request info
- `start_response` — a callback to set status + headers

And returns an **iterable of bytes** (the response body).

In [ ]:
# ============================================================
# Minimal WSGI App — This IS the entire interface
# ============================================================

def wsgi_app(environ, start_response):
    """A complete WSGI application in 6 lines."""
    # environ has: REQUEST_METHOD, PATH_INFO, QUERY_STRING, etc.
    path = environ.get('PATH_INFO', '/')
    method = environ.get('REQUEST_METHOD', 'GET')
    
    # Set response status and headers
    status = '200 OK'
    headers = [('Content-Type', 'text/plain')]
    start_response(status, headers)
    
    # Return body as iterable of bytes
    return [f"{method} {path} - Hello from WSGI!".encode('utf-8')]


# To run: gunicorn wsgi_app:wsgi_app
# Or use Python's built-in server:
# from wsgiref.simple_server import make_server
# server = make_server('localhost', 8000, wsgi_app)
# server.serve_forever()

# Let's test it ourselves:
class FakeStartResponse:
    def __call__(self, status, headers):
        print(f"Status: {status}")
        print(f"Headers: {headers}")

fake_environ = {'REQUEST_METHOD': 'GET', 'PATH_INFO': '/users'}
response = wsgi_app(fake_environ, FakeStartResponse())
print(f"Body: {response[0].decode()}")

In [ ]:
# ============================================================
# WSGI with routing — slightly more realistic
# ============================================================

def wsgi_app_with_routing(environ, start_response):
    path = environ.get('PATH_INFO', '/')
    method = environ.get('REQUEST_METHOD', 'GET')
    
    # Simple routing
    routes = {
        ('GET', '/'): ('200 OK', 'Welcome to WSGI'),
        ('GET', '/users'): ('200 OK', '{"users": ["alice", "bob"]}'),
        ('POST', '/users'): ('201 Created', '{"message": "User created"}'),
    }
    
    key = (method, path)
    if key in routes:
        status, body = routes[key]
        content_type = 'application/json' if body.startswith('{') else 'text/plain'
    else:
        status = '404 Not Found'
        body = '{"error": "Not found"}'
        content_type = 'application/json'
    
    headers = [('Content-Type', content_type)]
    start_response(status, headers)
    return [body.encode('utf-8')]

# This is essentially what Flask does under the hood!
# Flask wraps this pattern with decorators and routing.
print("WSGI = synchronous function call per request")
print("That's it. Flask/Django just add nice routing on top.")

## ⚡ ASGI — The Async Standard

ASGI apps are **async callables** that take three arguments:
- `scope` — connection info (type, path, method, headers)
- `receive` — async callable to receive messages (request body)
- `send` — async callable to send messages (response)

```
╔════════════════════════════════════════════════════════════════╗
║  WSGI                        ASGI                             ║
╠════════════════════════════════════════════════════════════════╣
║  def app(environ,            async def app(scope,             ║
║          start_response):            receive, send):          ║
║      body = do_work()            body = await do_work()       ║
║      return [body]               await send(body)             ║
║                                                                ║
║  Synchronous                 Asynchronous                     ║
║  One request at a time       Multiple concurrent requests     ║
║  Blocks on I/O               Yields during I/O                ║
║  Gunicorn                    Uvicorn / Hypercorn              ║
║  Flask, Django               FastAPI, Starlette               ║
╚════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Minimal ASGI App — The async version
# ============================================================
import json

async def asgi_app(scope, receive, send):
    """A complete ASGI application."""
    # scope contains: type, path, method, headers, query_string
    assert scope['type'] == 'http'
    
    path = scope.get('path', '/')
    method = scope.get('method', 'GET')
    
    # Read request body (if any)
    body = b''
    while True:
        message = await receive()
        body += message.get('body', b'')
        if not message.get('more_body', False):
            break
    
    # Prepare response
    response_body = json.dumps({
        "message": f"Hello from ASGI! {method} {path}"
    }).encode('utf-8')
    
    # Send response headers
    await send({
        'type': 'http.response.start',
        'status': 200,
        'headers': [
            [b'content-type', b'application/json'],
        ],
    })
    
    # Send response body
    await send({
        'type': 'http.response.body',
        'body': response_body,
    })

# To run: uvicorn asgi_app:asgi_app
print("ASGI is more verbose but enables:")
print("  • Async request handling")
print("  • WebSocket support")
print("  • HTTP/2 support")
print("  • Background tasks")
print("\nFastAPI/Starlette wrap this ugly code into clean decorators!")

## 🏭 Web Servers — Gunicorn vs Uvicorn

```
╔═══════════════════════════════════════════════════════════════════╗
║  SERVER COMPARISON                                               ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  GUNICORN (Green Unicorn)                                        ║
║  • WSGI server (sync)                                            ║
║  • Pre-fork worker model: spawns N worker processes              ║
║  • Each worker handles 1 request at a time                       ║
║  • gunicorn app:app -w 4 -b 0.0.0.0:8000                       ║
║  • Workers = 2 * CPU cores + 1 (rule of thumb)                  ║
║                                                                   ║
║  UVICORN                                                         ║
║  • ASGI server (async)                                           ║
║  • Single process, event loop (like Node.js)                     ║
║  • Handles thousands of concurrent connections                   ║
║  • uvicorn app:app --host 0.0.0.0 --port 8000                  ║
║                                                                   ║
║  PRODUCTION COMBO:                                                ║
║  gunicorn app:app -w 4 -k uvicorn.workers.UvicornWorker         ║
║  → Gunicorn manages processes, Uvicorn handles async per worker ║
║  → Best of both worlds!                                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Why Async Matters — The Performance Story
# ============================================================
import asyncio
import time

# Simulating I/O-bound work (DB queries, API calls)

# === SYNC approach (WSGI) ===
def sync_handler():
    """Each request blocks the thread."""
    time.sleep(0.1)  # Simulate DB query
    return "done"

# 5 requests sequentially = 0.5 seconds
start = time.time()
results = [sync_handler() for _ in range(5)]
sync_time = time.time() - start
print(f"Sync (5 requests): {sync_time:.2f}s")

# === ASYNC approach (ASGI) ===
async def async_handler():
    """Yields control during I/O wait."""
    await asyncio.sleep(0.1)  # Non-blocking!
    return "done"

async def handle_all():
    # 5 requests concurrently = ~0.1 seconds
    return await asyncio.gather(*[async_handler() for _ in range(5)])

start = time.time()
results = asyncio.run(handle_all())
async_time = time.time() - start
print(f"Async (5 requests): {async_time:.2f}s")
print(f"\nAsync is ~{sync_time/async_time:.0f}x faster for I/O-bound work!")

## 🤔 When to Use What?

```
╔═══════════════════════════════════════════════════════════════════╗
║  DECISION GUIDE                                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Use WSGI (Flask/Django + Gunicorn) when:                        ║
║  • CPU-bound work (image processing, ML inference)               ║
║  • Simple CRUD apps                                              ║
║  • Team is familiar with sync Python                             ║
║  • Using sync-only libraries                                     ║
║                                                                   ║
║  Use ASGI (FastAPI + Uvicorn) when:                              ║
║  • I/O-bound work (many DB/API calls)                           ║
║  • Need WebSocket support                                        ║
║  • High concurrency requirements                                 ║
║  • Real-time features (chat, notifications)                      ║
║  • Microservices calling other services                          ║
║                                                                   ║
║  RULE OF THUMB: New project? Use FastAPI + ASGI.                 ║
║  It handles both sync and async, and you get free docs.          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 🗺️ WSGI vs ASGI Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  WSGI / ASGI QUICK REFERENCE                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  WSGI: def app(environ, start_response) → [bytes]               ║
║  ASGI: async def app(scope, receive, send)                       ║
║                                                                   ║
║  WSGI servers: Gunicorn, uWSGI, mod_wsgi                        ║
║  ASGI servers: Uvicorn, Hypercorn, Daphne                       ║
║                                                                   ║
║  Production: gunicorn -k uvicorn.workers.UvicornWorker           ║
║  Workers: 2 * CPU_CORES + 1                                      ║
║                                                                   ║
║  ASGI advantages: async/await, WebSocket, HTTP/2                 ║
║  WSGI advantage: simpler, more battle-tested libraries          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is WSGI? | Standard interface between web servers and Python apps. Sync, callable-based |
| 2 | What is ASGI? | Async version of WSGI. Supports async/await, WebSocket, HTTP/2 |
| 3 | Why does FastAPI use ASGI? | Async request handling, WebSocket support, higher concurrency |
| 4 | Gunicorn vs Uvicorn? | Gunicorn = WSGI (sync, multi-process). Uvicorn = ASGI (async, event loop) |
| 5 | How to deploy FastAPI in production? | `gunicorn -k uvicorn.workers.UvicornWorker` — process management + async |
| 6 | Sync vs async for CPU-bound work? | Both are similar — async doesn't help CPU-bound work. Use multiprocessing |
| 7 | How many Gunicorn workers? | 2 * CPU_CORES + 1 (rule of thumb) |
| 8 | Can Django run on ASGI? | Yes! Django 3.0+ supports ASGI with Daphne/Uvicorn |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • WSGI = sync interface. ASGI = async interface        │
## │  • WSGI app: def(environ, start_response) → [bytes]    │
## │  • ASGI app: async def(scope, receive, send)            │
## │  • FastAPI → Starlette → ASGI → Uvicorn                │
## │  • Production: Gunicorn + Uvicorn workers               │
## │  • Async shines for I/O-bound, not CPU-bound            │
## │  • New project? Default to FastAPI + ASGI               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **API Best Practices** — Error handling, pagination, and production patterns